# Line Following — Part 1: Data Collection & Training

This notebook guides you through:
1. Collecting labelled images by **driving the robot with WASD** while the camera saves frames
2. Training a lightweight CNN (MobileNetV2) to predict steering direction
3. Saving the trained model weights as `line_follower.pth`

**Controls during data collection:**
| Key | Action | Label saved |
|-----|--------|-------------|
| W | Forward | `forward` |
| A | Turn left | `left` |
| D | Turn right | `right` |
| S | Stop | *(no image saved)* |

> Images are labelled **by what key you are pressing**, so drive naturally along the track and the labels will be correct automatically.

## Step 1 — Start the Camera

In [ ]:
import traitlets
import cv2
import numpy as np
import pyzed.sl as sl
import threading
import os
from traitlets.config.configurable import SingletonConfigurable

class Camera(SingletonConfigurable):
    color_value = traitlets.Any()

    def __init__(self):
        super(Camera, self).__init__()
        self.zed = sl.Camera()
        init_params = sl.InitParameters()
        init_params.camera_resolution = sl.RESOLUTION.VGA  # 672x376
        init_params.camera_fps = 100
        init_params.depth_mode = sl.DEPTH_MODE.NONE         # No depth needed
        init_params.coordinate_units = sl.UNIT.MILLIMETER
        status = self.zed.open(init_params)
        if status != sl.ERROR_CODE.SUCCESS:
            print("Camera Open:", repr(status))
            self.zed.close()
            exit(1)
        self.runtime = sl.RuntimeParameters()
        self.thread_runnning_flag = False
        camera_info = self.zed.get_camera_information()
        self.width  = camera_info.camera_configuration.resolution.width
        self.height = camera_info.camera_configuration.resolution.height
        self.image  = sl.Mat(self.width, self.height, sl.MAT_TYPE.U8_C4, sl.MEM.CPU)

    def _capture_frames(self):
        while self.thread_runnning_flag:
            if self.zed.grab(self.runtime) == sl.ERROR_CODE.SUCCESS:
                self.zed.retrieve_image(self.image, sl.VIEW.LEFT)
                bgra = self.image.get_data()
                self.color_value = cv2.cvtColor(bgra, cv2.COLOR_BGRA2BGR)

    def start(self):
        if not self.thread_runnning_flag:
            self.thread_runnning_flag = True
            self.thread = threading.Thread(target=self._capture_frames)
            self.thread.start()

    def stop(self):
        if self.thread_runnning_flag:
            self.thread_runnning_flag = False
            self.thread.join()

def bgr8_to_jpeg(value):
    return bytes(cv2.imencode('.jpg', value)[1])

camera = Camera()
camera.start()
print(f"Camera started: {camera.width}x{camera.height}")

## Step 2 — Collect Training Images with WASD Control

- Click inside the **Input text box** that appears below
- Type WASD keys to drive the robot — **each keypress saves a labelled image**
- Drive the full track multiple times, making sure to cover all sharp turns
- Aim for **at least 50 images per class** (left / forward / right)
- Press **S** to stop the robot (no image saved while stopped)
- When done, run the **Stop Collection** cell below

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import motors

# Create dataset folders
CLASS_DIRS = {
    'forward': 'dataset/forward',
    'left':    'dataset/left',
    'right':   'dataset/right',
}
for d in CLASS_DIRS.values():
    os.makedirs(d, exist_ok=True)

robot = motors.MotorsYukon(mecanum=False)

# Counters per class
counts = {'forward': 0, 'left': 0, 'right': 0}

# ── Display widgets ────────────────────────────────────────────────────────────
display_widget = widgets.Image(format='jpeg', width='45%')
count_label    = widgets.Label(value='forward: 0  |  left: 0  |  right: 0')
text_input     = widgets.Text(
    value='',
    placeholder='Click here then type WASD to drive',
    description='Input:',
    disabled=False
)
display(widgets.VBox([display_widget, count_label, text_input]))

# ── Live camera preview ────────────────────────────────────────────────────────
preview_count = 0
def on_camera_change(change):
    global preview_count
    preview_count += 1
    if preview_count % 2 == 0:  # update display every 2nd frame
        frame = change['new']
        if frame is not None:
            preview = cv2.resize(frame, None, fx=0.3, fy=0.3)
            display_widget.value = bgr8_to_jpeg(preview)

camera.observe(on_camera_change, names=['color_value'])

# ── WASD keyboard handler ──────────────────────────────────────────────────────
def save_frame(label):
    """Save the current camera frame to the correct class folder."""
    frame = camera.color_value
    if frame is None:
        return
    idx      = counts[label]
    filename = os.path.join(CLASS_DIRS[label], f'{label}_{idx:05d}.jpg')
    cv2.imwrite(filename, frame)
    counts[label] += 1
    count_label.value = (f"forward: {counts['forward']}  |  "
                         f"left: {counts['left']}  |  "
                         f"right: {counts['right']}")

def on_text_change(change):
    input_value = change['new']
    if not input_value:
        return

    key = input_value[-1].lower()  # use only the most recent character

    if key == 'w':
        robot.forward(0.35)
        save_frame('forward')
    elif key == 'a':
        robot.left(0.30)
        save_frame('left')
    elif key == 'd':
        robot.right(0.30)
        save_frame('right')
    elif key == 's':
        robot.stop()  # stop only — no image saved
    else:
        robot.stop()

text_input.observe(on_text_change, names='value')
print("Ready! Click the Input box and use WASD to drive and collect images.")

### Stop Collection
Run this cell when you have enough images.

In [ ]:
camera.unobserve(on_camera_change, names=['color_value'])
text_input.unobserve(on_text_change, names='value')
robot.stop()
print("Collection stopped.")
print(f"  forward : {counts['forward']} images")
print(f"  left    : {counts['left']} images")
print(f"  right   : {counts['right']} images")
print(f"  total   : {sum(counts.values())} images")

## Step 3 — Train MobileNetV2

Fine-tunes **MobileNetV2** (pretrained on ImageNet) with a 3-class head.  
Only the classifier layers are trained to keep it fast on the Jetson.

> Expected training time: ~5–10 minutes for 200 images over 10 epochs.

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset, random_split
import torch.nn as nn
import cv2
import numpy as np
import os
from PIL import Image as PILImage

DATASET_ROOT = 'dataset'
MODEL_PATH   = 'line_follower.pth'
BATCH_SIZE   = 16
EPOCHS       = 20
LR           = 3e-4
IMG_SIZE     = 224

# ── Yellow HSV range ──────────────────────────────────────────────────────────
YELLOW_LOWER = np.array([20, 100, 100])
YELLOW_UPPER = np.array([35, 255, 255])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

def get_steering_label(img_path):
    """
    Computes a steering value from -1.0 (full left) to 1.0 (full right)
    based on where the yellow line centroid sits in the bottom third of the image.
    Returns None if no line is detected.
    """
    img = cv2.imread(img_path)
    if img is None:
        return None
    h, w = img.shape[:2]
    roi  = img[int(h * 0.65):, :]
    hsv  = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, YELLOW_LOWER, YELLOW_UPPER)
    kernel = np.ones((5, 5), np.uint8)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) < 200:
        return None
    M = cv2.moments(largest)
    if M['m00'] == 0:
        return None
    cx     = M['m10'] / M['m00']
    norm_x = cx / w                  # 0.0 (left) to 1.0 (right)
    return (norm_x - 0.5) * 2.0     # remap to -1.0 to 1.0

# ── Build dataset with continuous steering labels ─────────────────────────────
class SteeringDataset(Dataset):
    def __init__(self, samples, transform):
        self.samples   = samples   # list of (img_path, steering_value)
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = PILImage.open(path).convert('RGB')
        return self.transform(img), torch.tensor([label], dtype=torch.float32)

# Scan all images across all class folders and auto-label by centroid position
all_samples = []
skipped = 0
for cls_dir in ['dataset/forward', 'dataset/left', 'dataset/right']:
    if not os.path.exists(cls_dir):
        continue
    for fname in os.listdir(cls_dir):
        if not fname.endswith('.jpg'):
            continue
        path    = os.path.join(cls_dir, fname)
        steering = get_steering_label(path)
        if steering is None:
            skipped += 1
            continue
        all_samples.append((path, steering))

print(f"Total usable images: {len(all_samples)}  (skipped {skipped} with no line)")
print(f"Steering range: {min(s for _,s in all_samples):.2f} to {max(s for _,s in all_samples):.2f}")

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

import random
random.shuffle(all_samples)
n_val      = max(1, int(0.2 * len(all_samples)))
n_train    = len(all_samples) - n_val
train_data = SteeringDataset(all_samples[:n_train], train_tf)
val_data   = SteeringDataset(all_samples[n_train:], val_tf)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ── Model — regression head ───────────────────────────────────────────────────
model = torchvision.models.mobilenet_v2(weights='IMAGENET1K_V1')
for i, param in enumerate(model.features.parameters()):
    param.requires_grad = True if i > 100 else False
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 1),  # single output: steering value
    nn.Tanh()                           # clamps output to -1.0 to 1.0
)
model = model.to(device)

criterion = nn.MSELoss()   # regression loss instead of CrossEntropy
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)

# ── Training loop ─────────────────────────────────────────────────────────────
best_val_loss = float('inf')

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs   = model(images)
            val_loss += criterion(outputs, labels).item()

    avg_train = running_loss / len(train_loader)
    avg_val   = val_loss    / len(val_loader)
    scheduler.step()

    print(f"Epoch [{epoch+1}/{EPOCHS}]  "
          f"Train Loss: {avg_train:.4f}  "
          f"Val Loss: {avg_val:.4f}")

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  ✓ Best model saved (val loss: {avg_val:.4f})")

print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")
print(f"Model saved to: {MODEL_PATH}")

In [7]:
import os
from PIL import Image as PILImage
import random
import torch
import torchvision
import torchvision.transforms as transforms
import torch.nn as nn
from torchvision.datasets import ImageFolder

DATASET_ROOT = 'dataset'
MODEL_PATH   = 'line_follower.pth'
IMG_SIZE     = 224

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

full_dataset = ImageFolder(
    root=DATASET_ROOT,
    is_valid_file=lambda x: x.endswith('.jpg') and '.ipynb_checkpoints' not in x
)
CLASS_NAMES  = full_dataset.classes
CLASS_DIRS   = {
    'forward': 'dataset/forward',
    'left':    'dataset/left',
    'right':   'dataset/right',
}

# Reload model
model = torchvision.models.mobilenet_v2(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(model.last_channel, 1),
    nn.Tanh()
)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

# Verification
sample_paths = []
for cls_name, cls_dir in CLASS_DIRS.items():
    files = [os.path.join(cls_dir, f) for f in os.listdir(cls_dir)
             if f.endswith('.jpg') and '.ipynb_checkpoints' not in f]
    if files:
        sample_paths += random.sample(files, min(3, len(files)))

for path in sample_paths:
    img_pil = PILImage.open(path).convert('RGB')
    x = val_tf(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        steering = model(x).item()
    true_label = os.path.basename(os.path.dirname(path))
    print(f"True: {true_label:>8}  |  Steering: {steering:+.3f}  "
          f"({'LEFT' if steering < -0.1 else 'RIGHT' if steering > 0.1 else 'STRAIGHT'})")

/tmp/ipykernel_2630/137849706.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(MODEL_PATH, map_location=device))


True:  forward  |  Steering: +0.038  (STRAIGHT)
True:  forward  |  Steering: -0.228  (LEFT)
True:  forward  |  Steering: +0.068  (STRAIGHT)
True:     left  |  Steering: -0.278  (LEFT)
True:     left  |  Steering: +0.001  (STRAIGHT)
True:     left  |  Steering: -0.281  (LEFT)
True:    right  |  Steering: -0.240  (LEFT)
True:    right  |  Steering: +0.154  (RIGHT)
True:    right  |  Steering: +0.538  (RIGHT)
